# Прогон ДЗ: RAG по ТФКП

Ноутбук только запускает скрипты. Сдаются `build_index.py` и `predict.py`, не этот файл.

Runtime: CPU, Python 3.11. В Secrets заранее положите `LLM_BASE_URL`, `LLM_MODEL`, `LLM_API_KEY`, `SUBMISSION_ID`. Модель endpoint должна быть Qwen3.5-4B.

Откройте ноутбук из корня проекта, рядом с `build_index.py`.

In [ ]:
from pathlib import Path
import sys
if not Path('build_index.py').exists():
    raise SystemExit('Запускайте из корня проекта, рядом с build_index.py')
!{sys.executable} -m pip install -q -r requirements.txt
# В Colab torchvision собран под другой torch. Нам он не нужен и ломает import BERT.
!{sys.executable} -m pip uninstall -y torchvision torchaudio

In [ ]:
import os
try:
    from google.colab import userdata
except ImportError:
    userdata = None

def secret(name):
    if userdata is not None:
        value = userdata.get(name)
        if value:
            return value
    if os.environ.get(name):
        return os.environ[name]
    raise RuntimeError(f'Нет {name}')

for name in ('LLM_BASE_URL', 'LLM_MODEL', 'LLM_API_KEY', 'SUBMISSION_ID'):
    os.environ[name] = secret(name)
print('endpoint задан, модель:', os.environ['LLM_MODEL'])

In [ ]:
import os
from openai import OpenAI

try:
    from google.colab import userdata
except ImportError:
    userdata = None

for name in ('LLM_BASE_URL', 'LLM_MODEL', 'LLM_API_KEY', 'SUBMISSION_ID'):
    if userdata is not None:
        value = userdata.get(name)
        if value:
            os.environ[name] = value
    if not os.environ.get(name):
        raise SystemExit(f'Нет секрета {name}')

print('проверка', os.environ['LLM_MODEL'], flush=True)
client = OpenAI(
    base_url=os.environ['LLM_BASE_URL'].rstrip('/'),
    api_key=os.environ['LLM_API_KEY'],
    timeout=90,
)
try:
    response = client.chat.completions.create(
        model=os.environ['LLM_MODEL'],
        messages=[{'role': 'user', 'content': 'Ответь одним словом: ок'}],
        max_tokens=16,
        temperature=0.7,
        top_p=0.8,
        presence_penalty=1.5,
        extra_body={
            'top_k': 20,
            'min_p': 0,
            'repetition_penalty': 1.0,
            'chat_template_kwargs': {'enable_thinking': False},
        },
        extra_headers={
            'X-Question-ID': 'ping',
            'X-Submission-ID': os.environ['SUBMISSION_ID'],
        },
    )
except Exception as exc:
    raise SystemExit('Модель не вызвалась. Длинную ячейку не запускайте.\n' + str(exc))
text = (response.choices[0].message.content or '').strip()
print('модель ответила:', text[:300])

In [ ]:
import urllib.request
from pathlib import Path
import sys
Path('data').mkdir(exist_ok=True)
pdf = Path('data/tfkp.pdf')
url = 'https://new.mipt.ru/upload/mathematics/study/uchebniki/%D0%9B_%D0%A2%D0%A4%D0%9A%D0%9F_%D0%93%D0%BE%D1%80%D1%8F%D0%B9%D0%BD%D0%BE%D0%B2_%D0%9F%D0%BE%D0%BB%D0%BE%D0%B2%D0%B8%D0%BD%D0%BA%D0%B8%D0%BD_2025.pdf'
if not pdf.exists():
    urllib.request.urlretrieve(url, pdf)
!{sys.executable} extract_corpus.py --pdf data/tfkp.pdf --output data/corpus.jsonl

In [ ]:
import sys
!{sys.executable} build_index.py --corpus data/corpus.jsonl --output index --config config.json

In [ ]:
import subprocess
import sys
from pathlib import Path
Path('runs').mkdir(exist_ok=True)
systems = ['no_context', 'single', 'final', 'no_rerank', 'no_dense']
for name in systems:
    print('===', name, flush=True)
    subprocess.run([
        sys.executable, 'predict.py',
        '--index', 'index',
        '--questions', 'dev_questions.jsonl',
        '--output', f'runs/{name}_answers.jsonl',
        '--trace', f'runs/{name}_trace.jsonl',
        '--config', 'config.json',
        '--system', name,
    ], check=True)

In [ ]:
import shutil
import sys
shutil.copyfile('runs/final_answers.jsonl', 'dev_answers.jsonl')
shutil.copyfile('runs/final_trace.jsonl', 'dev_trace.jsonl')
!{sys.executable} collect_experiments.py --gold dev_gold.jsonl --runs runs --csv experiments.csv --report report.md --corpus data/corpus.jsonl --stats index/stats.json